# Phase 3 — LoRA Fine-Tuning — intent-classifier

This notebook trains and evaluates LoRA adapters for all 5 models × 4 configs on Google Colab.

**What runs here:**
1. Clone the repo from GitHub
2. Install dependencies
3. HF authentication (token stored in Colab Secrets)
4. GPU environment check
5. Configuration
6. Prepare data splits
7. Smoke test — 10 steps to validate the pipeline
8. Full training — all (model × config) combinations
9. Validation — load each adapter, evaluate on val split
10. Test evaluation (locked — run deliberately)
11. Inference sanity check
12. Download reports (zip)

**Note:** Plot generation is done offline. Reports are JSON files — download them,
then run `plot_lora_results.py` locally.

## 1. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

print(f"Repo at: {REPO_DIR}")

## 2. Install dependencies

In [ ]:
%pip install -q \
    torch \
    torchao \
    transformers \
    accelerate \
    peft \
    trl \
    datasets \
    bitsandbytes \
    huggingface_hub \
    python-dotenv \
    sentencepiece \
    protobuf

print("Dependencies installed.")

## 3. Hugging Face authentication

Store your `HF_TOKEN` in **Colab Secrets** (key icon in the left sidebar).
Required for:
- Downloading `meta-llama/Llama-3.2-1B-Instruct` (gated model)
- Uploading trained adapters to `kon172verma/intent-classifier`

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("WARNING: HF_TOKEN secret is empty.")
except Exception as e:
    print(f"Not running in Colab or secret missing: {e}")

## 4. GPU environment check

In [ ]:
import subprocess
import platform
import torch

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("GPU     : not available — training will be very slow on CPU")

## 5. Configuration

Edit the variables below to control the experiment matrix.

| Variable | Description |
|---|---|
| `MODELS_TO_RUN` | Which models to train (subset or all 5) |
| `CONFIGS_TO_RUN` | Which LoRA configs (A / B / C / D) |
| `DATASET_SIZE` | `"1k"` or `"10k"` |
| `DEVICE` | `"auto"` (recommended) or `"cuda"` |

**LoRA config legend:**
| Config | Description |
|---|---|
| A | Light — Q+V only, r=8 |
| B | Standard — full attention, r=16 |
| C | Wide — attn+MLP, r=16 |
| D | Heavy — attn+MLP, r=32 |

In [ ]:
import sys

REPO_DIR  = "/content/intent-classifier"
SRC_DIR   = f"{REPO_DIR}/finetune_LoRA/src"
DATA_DIR  = f"{REPO_DIR}/finetune_LoRA/data"
SPLIT_DIR = f"{DATA_DIR}/1k"          # updated per DATASET_SIZE below

# ── Experiment matrix ──────────────────────────────────────────────────────
ALL_MODELS = [
    "smollm2-360m",
    "qwen2.5-0.5b",
    "qwen3-0.6b",
    "llama3.2-1b",
    "smollm2-1.7b",
]
ALL_CONFIGS = ["A", "B", "C", "D"]

MODELS_TO_RUN  = ALL_MODELS   # change to a subset for partial runs
CONFIGS_TO_RUN = ALL_CONFIGS  # change to e.g. ["A", "B"] for partial

DATASET_SIZE = "1k"           # "1k" or "10k"
DEVICE       = "auto"

# Keep SPLIT_DIR in sync
SPLIT_DIR = f"{DATA_DIR}/{DATASET_SIZE}"

# Add repo root to path so finetune_lib / evaluation_lib are importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Configuration:")
print(f"  Models   : {MODELS_TO_RUN}")
print(f"  Configs  : {CONFIGS_TO_RUN}")
print(f"  Dataset  : {DATASET_SIZE}")
print(f"  Device   : {DEVICE}")

## 6. Prepare data splits

In [ ]:
import os
import subprocess
import sys

os.makedirs(SPLIT_DIR, exist_ok=True)

PREP_SCRIPT = f"{REPO_DIR}/finetune_LoRA/src/prepare_lora_data.py"
cmd = [
    sys.executable, "-u", PREP_SCRIPT,
    "--dataset-size", DATASET_SIZE,
    "--out-dir",      DATA_DIR,
]
print("Preparing data splits...")
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("prepare_lora_data.py failed")
print("Data preparation complete.")


## 7. Smoke test

Runs **10 training steps** on the first model + config A to validate the
complete pipeline (data loading → tokenisation → PEFT → training → report save)
without committing to a full run.

**Safe to skip** if you have already validated the pipeline.

In [ ]:
import subprocess
import sys
import time

SMOKE_MODEL  = MODELS_TO_RUN[0]
SMOKE_CONFIG = "A"

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/lora_train.py",
    "--model",        SMOKE_MODEL,
    "--lora-config",  SMOKE_CONFIG,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--smoke-test",
    "--no-push",
]
print("Running smoke test:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nSmoke test {'PASSED' if proc.returncode == 0 else 'FAILED'} in {elapsed:.0f}s")


## 8. Train

Runs all (model × config) combinations via `run_lora_experiments.py`.

After each run the adapter is:
1. Saved locally → `finetune_LoRA/adapters/{run_tag}/`
2. Pushed to HF  → `kon172verma/intent-classifier/LoRA/{run_tag}/`

Training reports (JSON with full `log_history`) are saved to
`finetune_LoRA/reports_training/`.

**Estimated runtimes on L4 GPU (1k dataset):**

| Model | Config A | Config B | Config C | Config D |
|---|---|---|---|---|
| smollm2-360m   | ~8 min  | ~10 min | ~12 min | ~15 min |
| qwen2.5-0.5b   | ~10 min | ~13 min | ~15 min | ~18 min |
| qwen3-0.6b     | ~10 min | ~13 min | ~15 min | ~18 min |
| llama3.2-1b    | ~20 min | ~25 min | ~28 min | ~35 min |
| smollm2-1.7b   | ~25 min | ~30 min | ~35 min | ~42 min |

In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/run_lora_experiments.py",
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
]
print("Running:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nTraining {'complete' if proc.returncode == 0 else 'FAILED'} in {elapsed/60:.1f} min")


## 9. Validation

Evaluates each adapter on the validation split.
Reports are saved to `finetune_LoRA/reports_validation/`.

In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/run_lora_experiments.py",
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--skip-training",  # adapters already uploaded; load from HF
]
print("Running validation:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
print(f"\nValidation {'complete' if proc.returncode == 0 else 'FAILED'} in {time.time()-t0:.0f}s")


## 10. Test evaluation (locked)

Run this cell deliberately **once** after reviewing validation results.

For `DATASET_SIZE = "1k"`, the test set is `data/1k/test.jsonl`.
For `DATASET_SIZE = "10k"`, the anchor slice (first 100 of `test.jsonl`) aligns
with `sample_0001` for cross-experiment comparison.

Reports saved to `finetune_LoRA/reports_test/`.

In [ ]:
import subprocess
import sys
import time

for model_key in MODELS_TO_RUN:
    for cfg in CONFIGS_TO_RUN:
        run_tag = f"{model_key}_{cfg}_{DATASET_SIZE}"
        cmd = [
            sys.executable, "-u", f"{SRC_DIR}/lora_validate.py",
            "--model",        model_key,
            "--lora-config",  cfg,
            "--dataset-size", DATASET_SIZE,
            "--split",        "test",
            "--device",       DEVICE,
            # loads from HF Hub by default
        ]
        print(f"\n  Testing {run_tag}...")
        t0 = time.time()
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=REPO_DIR,
        )
        _buf: list[str] = []
        while True:
            ch = proc.stdout.read(1)
            if not ch:
                if _buf:
                    line = "".join(_buf)
                    if not line.startswith("Failed to load "):
                        print(line, end="", flush=True)
                break
            _buf.append(ch)
            if ch in ("\r", "\n"):
                line = "".join(_buf)
                if not line.startswith("Failed to load "):
                    print(line, end="", flush=True)
                _buf = []
        proc.stdout.close()
        proc.wait()
        print(f"  {'OK' if proc.returncode == 0 else 'FAILED'} in {time.time()-t0:.0f}s")


## 11. Inference sanity check

Manually inspect model outputs on a few hand-picked examples.
This catches output format issues (e.g., thinking tokens leaking through) that
aggregate accuracy metrics would not surface.

In [ ]:
import json
import sys
import torch
from pathlib import Path

# Pick any run_tag to inspect
INSPECT_MODEL  = MODELS_TO_RUN[0]
INSPECT_CONFIG = CONFIGS_TO_RUN[0]

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from finetune_lib import (
    FINETUNE_MODEL_REGISTRY, HF_HUB_REPO, hf_adapter_subfolder,
    build_chat_messages, apply_chat_template_safe, extract_prediction,
    load_jsonl,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = FINETUNE_MODEL_REGISTRY[INSPECT_MODEL]
hf_sub   = hf_adapter_subfolder("LoRA", INSPECT_MODEL, INSPECT_CONFIG, DATASET_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if device.type == "cuda" else torch.float32

tokenizer  = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map={"": device})
model      = PeftModel.from_pretrained(base_model, HF_HUB_REPO, subfolder=hf_sub)
model.eval()

val_file = Path(f"{DATA_DIR}/{DATASET_SIZE}/val.jsonl")
examples = load_jsonl(val_file)[:5]  # first 5 examples

for i, ex in enumerate(examples, 1):
    msgs = build_chat_messages(ex, include_answer=False)
    text = apply_chat_template_safe(tokenizer, msgs, INSPECT_MODEL, add_generation_prompt=True)
    inp  = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=16, do_sample=False,
                             pad_token_id=int(tokenizer.eos_token_id))
    new_ids = out[0][inp["input_ids"].shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True)
    pred    = extract_prediction(raw)
    correct = pred == ex["answer"]
    print(f"[{i}] query={ex['query'][:60]!r}")
    print(f"     expected={ex['answer']!r}  got={pred!r}  raw={raw!r}  {'✓' if correct else '✗'}")
    print()

## 12. Download reports

Zips and downloads all JSON report files so you can run plot scripts locally.

Contents of the zip:
- `reports_training/` — training curves + log history (one JSON per run)
- `reports_validation/` — validation accuracy (one JSON per run)
- `reports_test/` — test accuracy + per-tool metrics (one JSON per run)

In [ ]:
import os
import shutil
from pathlib import Path

try:
    from google.colab import files  # type: ignore

    lora_dir = Path(REPO_DIR) / "finetune_LoRA"
    zip_base = "/content/lora_reports"

    # Gather all report directories that exist
    report_dirs = {
        "reports_training":   lora_dir / "reports_training",
        "reports_validation": lora_dir / "reports_validation",
        "reports_test":       lora_dir / "reports_test",
    }

    # Build a temporary staging directory
    staging = Path("/content/_lora_reports_staging")
    staging.mkdir(exist_ok=True)
    for name, src in report_dirs.items():
        if src.exists() and any(src.iterdir()):
            shutil.copytree(src, staging / name, dirs_exist_ok=True)
        else:
            print(f"  Skipping {name} (empty or missing)")

    shutil.make_archive(zip_base, "zip", staging)
    print(f"Created {zip_base}.zip")
    files.download(f"{zip_base}.zip")

except ImportError:
    print("Not running in Colab — copy reports manually from:")
    for name in ["reports_training", "reports_validation", "reports_test"]:
        print(f"  {REPO_DIR}/finetune_LoRA/{name}/")